## Use Yang's formula and use P&S amplitude

In [35]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2, log10

# ---------- 文件路径 ----------
ASSIGN_FILE = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/assignmets_gamma_v3_original.csv'
MERGE_PICKS_FILE = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/phasenet_picks_2212raw_withamp.csv'
CATALOG_FILE = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/gamma_catalog.csv'
STATION_FILE = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/宽频带布点.txt'
RELOC_FILE = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/hypoDD_phasenet202212.reloc'

# ---------- 输出文件 ----------
OUT_GROUP_FILE = 'event_station_results.csv'
OUT_SUMMARY_FILE = 'event_magnitude_summary.csv'
OUT_UPDATED_RELOC = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/hypoDD_phasenet202212_withmag.reloc'

# ---------- 读取文件 ----------
assignments = pd.read_csv(ASSIGN_FILE)
# ---------- 读取文件 ----------
merge_picks = pd.read_csv(MERGE_PICKS_FILE)
merge_picks = merge_picks.dropna().reset_index(drop=True)
# ✅ 让 pick_index = DataFrame index（与 CSV 数据行号一致）
merge_picks['pick_index'] = merge_picks.index
print(merge_picks.head())
merge_picks['phase_amplitude'] = merge_picks['phase_amplitude']
# print(merge_picks.head(5))
catalog = pd.read_csv(CATALOG_FILE, sep='\t')
stations = pd.read_csv(STATION_FILE, delim_whitespace=True)
reloc = pd.read_csv(RELOC_FILE, delim_whitespace=True)

# ---------- 数据预处理 ----------
assignments['event_index'] = assignments['event_index'].astype(int)
stations['station_id'] = stations['station_id'].astype(str)
reloc['id'] = reloc['id'].astype(str)


# ---------- 定义函数 ----------
def haversine(lat1, lon1, lat2, lon2):
    """计算球面距离，单位 km"""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def calc_magnitude(amp, dist):
    """按给定公式计算震级"""
    if amp <= 0 or dist <= 0 or np.isnan(amp) or np.isnan(dist):
        return np.nan
    return log10(amp) + 1.26 * log10(dist) - 0.0026 * dist - 2.2

# ---------- 主计算 ----------
results = []
summary = []

for event_id, picks_in_event in assignments.groupby('event_index'):
    # 从 assignments 中获得该事件对应的 pick_index（实际上是 merge_picks 的行号）
    pick_indices = picks_in_event['pick_index'].astype(int).values

    # 在 merge_picks 中取出这些行
    event_picks = merge_picks[merge_picks['pick_index'].isin(pick_indices)]

    # 如果该事件没有匹配的拾取，跳过
    if event_picks.empty:
        continue

    # 找事件的震中位置
    event_info = catalog[catalog['event_index'] == event_id]
    if event_info.empty:
        continue
    ev_lat = float(event_info['latitude'])
    ev_lon = float(event_info['longitude'])

    # 按 station_id 分组，计算平均振幅
    M_group_list = []
    for station_id, group in event_picks.groupby('station_id'):
        mean_amp = group['phase_amplitude'].mean()

        # 查找台站坐标
        st_info = stations[stations['station_id'] == station_id]
        if st_info.empty:
            continue
        st_lat = float(st_info['latitude'])
        st_lon = float(st_info['longitude'])

        # 计算距离与震级
        dist = haversine(ev_lat, ev_lon, st_lat, st_lon)
        M_group = calc_magnitude(mean_amp, dist)

        results.append({
            'event_index': event_id,
            'station_id': station_id,
            'mean_amp': mean_amp,
            'dist_km': dist,
            'M_group': M_group
        })

        if not np.isnan(M_group):
            M_group_list.append(M_group)

    # 对所有 station_id 组的 M_group 求平均，得到事件最终震级
    if len(M_group_list) > 0:
        M_final = np.mean(M_group_list)-0.5
    else:
        M_final = np.nan

    summary.append({
        'event_index': event_id,
        'final_magnitude': M_final,
        'station_group_count': len(M_group_list)
    })

# ---------- 保存结果 ----------
df_results = pd.DataFrame(results)
df_summary = pd.DataFrame(summary)

# df_results.to_csv(OUT_GROUP_FILE, index=False, float_format='%.6f')
# df_summary.to_csv(OUT_SUMMARY_FILE, index=False, float_format='%.6f')

print(f"逐台站结果已保存: {OUT_GROUP_FILE}")
print(f"事件汇总结果已保存: {OUT_SUMMARY_FILE}")

# ---------- 将震级更新到 reloc 文件 ----------
# 将 event_index 转为字符串，与 reloc['id'] 对齐
df_summary['event_index'] = df_summary['event_index'].astype(str)
mapping = pd.Series(df_summary['final_magnitude'].values, index=df_summary['event_index']).to_dict()

# 替换 magnitude 列
def update_magnitude(row):
    eid = str(row['id'])
    if eid in mapping and not pd.isna(mapping[eid]):
        return mapping[eid]
    return row['magnitude']

reloc['magnitude'] = reloc.apply(update_magnitude, axis=1)
reloc['magnitude'] = reloc['magnitude']-1

# ---------- 输出更新后的 reloc 文件 ----------
reloc.to_csv(OUT_UPDATED_RELOC, sep=' ', index=False, float_format='%.6f')
print(f"✅ 更新后的 reloc 文件已保存: {OUT_UPDATED_RELOC}")

   index station_id                     time               start_time  \
0      0       QJ01  2022-12-01 01:06:14.240  2022-12-01 01:06:14.130   
1      1       QJ01  2022-12-01 01:06:16.840  2022-12-01 01:06:16.640   
2      2       QJ01  2022-12-01 01:09:23.480  2022-12-01 01:09:23.410   
3      3       QJ01  2022-12-01 01:47:22.800  2022-12-01 01:47:22.710   
4      4       QJ01  2022-12-01 02:23:13.390  2022-12-01 02:23:13.340   

                  end_time  probability phase  phase_amplitude  pick_index  
0  2022-12-01 01:06:14.410     0.495497     P           2080.0           0  
1  2022-12-01 01:06:17.060     0.576545     S           2292.0           1  
2  2022-12-01 01:09:23.630     0.391696     P           1952.0           2  
3  2022-12-01 01:47:22.960     0.450714     S           2923.0           3  
4  2022-12-01 02:23:13.540     0.358142     S           1265.0           4  


/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:28: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  stations = pd.read_csv(STATION_FILE, delim_whitespace=True)
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:29: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  reloc = pd.read_csv(RELOC_FILE, delim_whitespace=True)
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:73: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ev_lat = float(event_info['latitude'])
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:74: FutureWarning: Calling float on a single element Series is deprecated and will raise 

逐台站结果已保存: event_station_results.csv
事件汇总结果已保存: event_magnitude_summary.csv
✅ 更新后的 reloc 文件已保存: /Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/hypoDD_phasenet202212_withmag.reloc


/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:73: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ev_lat = float(event_info['latitude'])
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:74: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ev_lon = float(event_info['longitude'])
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:85: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  st_lat = float(st_info['latitude'])
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_50945/679467476.py:86: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(

## 用S波的振幅计算，use Yang's formula

In [21]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2, log10
import warnings

# ---------------- 用户可配置的文件名 ----------------
ASSIGN_FILE = 'assignmets_gamma_original.csv'
MERGE_PICKS_FILE = 'merged_picks.csv'
CATALOG_FILE = 'gamma_catalog.csv'
STATION_FILE = '宽频带布点.txt'
RELOC_FILE = 'hypoDD_v2_1307_withmag_modified_updated.reloc'

# 输出（使用不同文件名以避免覆盖）
OUT_GROUP_FILE = 'event_station_results_Sphase.csv'
OUT_SUMMARY_FILE = 'event_magnitude_summary_Sphase.csv'
OUT_UPDATED_RELOC = 'hypoDD_v2_1307_withmag_modified_final_Sphase.reloc'
# ----------------------------------------------------

def haversine(lat1, lon1, lat2, lon2):
    """两点球面距离（km）"""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2.0)**2 + cos(lat1)*cos(lat2)*sin(dlon/2.0)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def calc_magnitude(amp, dist):
    """按给定公式计算震级（以10为底）"""
    try:
        if amp is None or dist is None or np.isnan(amp) or np.isnan(dist):
            return np.nan
        if amp <= 0 or dist <= 0:
            return np.nan
        return log10(amp) + 1.26 * log10(dist) - 0.0026 * dist - 2.2
    except Exception:
        return np.nan


# 1) 读取文件（对 merge_picks 使用 reset_index() 把行号作为 pick_index）
assignments = pd.read_csv(ASSIGN_FILE)
try:
    merge_picks = pd.read_csv(MERGE_PICKS_FILE)
except Exception:
    merge_picks = pd.read_csv(MERGE_PICKS_FILE, sep=r'\s+', engine='python')
merge_picks = merge_picks.reset_index().rename(columns={'index': 'pick_index'})
merge_picks['pick_index'] = merge_picks['pick_index'].astype(int)

catalog = pd.read_csv(CATALOG_FILE, sep='\t', engine='python')
stations = pd.read_csv(STATION_FILE, delim_whitespace=True)
reloc = pd.read_csv(RELOC_FILE, delim_whitespace=True)

# 2) 类型规范
assignments['pick_index'] = pd.to_numeric(assignments['pick_index'], errors='coerce')
assignments = assignments[assignments['pick_index'].notna()].copy()
assignments['pick_index'] = assignments['pick_index'].astype(int)
assignments['event_index'] = assignments['event_index'].astype(str)
catalog['event_index'] = catalog['event_index'].astype(str)

merge_picks['station_id'] = merge_picks['station_id'].astype(str)
stations['station_id'] = stations['station_id'].astype(str)
stations['longitude'] = pd.to_numeric(stations['longitude'], errors='coerce')
stations['latitude'] = pd.to_numeric(stations['latitude'], errors='coerce')

# 检查必要列
if 'phase_amplitude' not in merge_picks.columns or 'phase_type' not in merge_picks.columns:
    raise RuntimeError("merge_picks 必须包含列 'phase_amplitude' 和 'phase_type'。")

# 3) 主计算
station_level_rows = []
event_summary_rows = []
stations_idx = stations.set_index('station_id')[['latitude', 'longitude']].to_dict(orient='index')

event_ids = assignments['event_index'].unique()
print(f"总共有 {len(event_ids)} 个在 assignments 中出现的 event_index 将被处理（unique）。")

for ev_id in event_ids:
    pick_indices = assignments.loc[assignments['event_index'] == ev_id, 'pick_index'].values
    picks_ev = merge_picks[merge_picks['pick_index'].isin(pick_indices)].copy()

    # 仅保留 phase_type == 'S' 的行
    picks_ev_S = picks_ev[picks_ev['phase_type'].str.upper() == 'S']
    if picks_ev_S.empty:
        warnings.warn(f"事件 {ev_id} 没有 S 相数据，跳过。")
        continue

    # 事件位置
    ev_row = catalog[catalog['event_index'] == ev_id]
    if ev_row.empty:
        warnings.warn(f"事件 {ev_id} 未在 catalog 中找到。")
        continue
    ev_lat = float(ev_row.iloc[0]['latitude'])
    ev_lon = float(ev_row.iloc[0]['longitude'])

    grouped = (
        picks_ev_S
        .groupby('station_id', as_index=False)['phase_amplitude']
        .mean()
        .rename(columns={'phase_amplitude': 'mean_amp'})
    )

    M_group_list = []
    for _, g in grouped.iterrows():
        station_id = str(g['station_id'])
        mean_amp = g['mean_amp']

        if station_id not in stations_idx:
            continue
        st = stations_idx[station_id]
        st_lat = float(st['latitude'])
        st_lon = float(st['longitude'])

        dist_km = haversine(ev_lat, ev_lon, st_lat, st_lon)
        M_group = calc_magnitude(mean_amp, dist_km)

        station_level_rows.append({
            'event_index': ev_id,
            'station_id': station_id,
            'mean_amp': mean_amp,
            'dist_km': dist_km,
            'M_group': M_group
        })

        if not np.isnan(M_group):
            M_group_list.append(M_group)

    if len(M_group_list) > 0:
        final_M = np.mean(M_group_list)
    else:
        final_M = np.nan

    event_summary_rows.append({
        'event_index': ev_id,
        'final_magnitude': final_M,
        'station_group_count': len(M_group_list)
    })

# 4) 保存结果
df_station_groups = pd.DataFrame(station_level_rows)
df_event_summary = pd.DataFrame(event_summary_rows)
df_station_groups.to_csv(OUT_GROUP_FILE, index=False, float_format='%.6f')
df_event_summary.to_csv(OUT_SUMMARY_FILE, index=False, float_format='%.6f')

print(f"逐台站结果（S相）已保存到: {OUT_GROUP_FILE}")
print(f"事件汇总结果（S相）已保存到: {OUT_SUMMARY_FILE}")
print(f"有效计算的事件数: {len(df_event_summary)}")

# ---------- 将震级更新到 reloc 文件 ----------
# 将 event_index 转为字符串，与 reloc['id'] 对齐
df_event_summary['event_index'] = df_event_summary['event_index'].astype(str)
mapping = pd.Series(df_event_summary['final_magnitude'].values, index=df_event_summary['event_index']).to_dict()

# 替换 magnitude 列
def update_magnitude(row):
    eid = str(row['id'])
    if eid in mapping and not pd.isna(mapping[eid]):
        return mapping[eid]
    return row['magnitude']

reloc['magnitude'] = reloc.apply(update_magnitude, axis=1)

# ---------- 输出更新后的 reloc 文件 ----------
reloc.to_csv(OUT_UPDATED_RELOC, sep=' ', index=False, float_format='%.6f')

print(f"✅ 已将 S 相计算的震级写回 reloc 并保存为: {OUT_UPDATED_RELOC}")



总共有 5169 个在 assignments 中出现的 event_index 将被处理（unique）。


/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_62893/859297841.py:88: UserWarning: 事件 900 没有 S 相数据，跳过。
  warnings.warn(f"事件 {ev_id} 没有 S 相数据，跳过。")
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_62893/859297841.py:88: UserWarning: 事件 1060 没有 S 相数据，跳过。
  warnings.warn(f"事件 {ev_id} 没有 S 相数据，跳过。")
/var/folders/y4/__h0s0zj0qb4gjkjzynnyrcr0000gn/T/ipykernel_62893/859297841.py:88: UserWarning: 事件 2481 没有 S 相数据，跳过。
  warnings.warn(f"事件 {ev_id} 没有 S 相数据，跳过。")


逐台站结果（S相）已保存到: event_station_results_Sphase.csv
事件汇总结果（S相）已保存到: event_magnitude_summary_Sphase.csv
有效计算的事件数: 5166
✅ 已将 S 相计算的震级写回 reloc 并保存为: hypoDD_v2_1307_withmag_modified_final_Sphase.reloc


### 用国家标准值再算一次，only use S amplitude

In [22]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2, log10
import warnings

# ---------------- 文件路径 ----------------
ASSIGN_FILE = 'assignmets_gamma_original.csv'
MERGE_PICKS_FILE = 'merged_picks.csv'
CATALOG_FILE = 'gamma_catalog.csv'
STATION_FILE = '宽频带布点.txt'
RELOC_FILE = 'hypoDD_v2_1307_withmag_modified_updated.reloc'

# 输出（带 Sphase 和 ML 标记）
OUT_GROUP_FILE = 'event_station_results_Sphase_ML.csv'
OUT_SUMMARY_FILE = 'event_magnitude_summary_Sphase_ML.csv'
OUT_UPDATED_RELOC = 'hypoDD_v2_1307_withmag_modified_final_Sphase_ML.reloc'
# ----------------------------------------------------

# 震中距—量规函数 R(dist) 对照表（单位 km）
R_table = {
    0: 2.0, 5: 2.0, 10: 2.0, 15: 2.1, 20: 2.2,
    25: 2.4, 30: 2.6, 35: 2.7, 40: 2.8, 45: 2.9,
    50: 3.0, 55: 3.1, 60: 3.2
}

def interpolate_R(dist):
    """线性插值 R(dist)"""
    if dist <= 0:
        return 2.0
    if dist >= 60:
        return 3.2
    keys = sorted(R_table.keys())
    for i in range(len(keys) - 1):
        d1, d2 = keys[i], keys[i + 1]
        if d1 <= dist <= d2:
            R1, R2 = R_table[d1], R_table[d2]
            return R1 + (R2 - R1) * (dist - d1) / (d2 - d1)
    return 3.2  # fallback

def haversine(lat1, lon1, lat2, lon2):
    """两点球面距离（km）"""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def calc_ML(amp, dist):
    """地方震级 ML = log10(A) + R(dist)"""
    if amp <= 0 or np.isnan(amp) or np.isnan(dist):
        return np.nan
    R_value = interpolate_R(dist)
    return log10(amp) + R_value

def main():
    # ---------- 1. 读取文件 ----------
    assignments = pd.read_csv(ASSIGN_FILE)
    try:
        merge_picks = pd.read_csv(MERGE_PICKS_FILE)
    except Exception:
        merge_picks = pd.read_csv(MERGE_PICKS_FILE, sep=r'\s+', engine='python')
    merge_picks = merge_picks.reset_index().rename(columns={'index': 'pick_index'})
    merge_picks['pick_index'] = merge_picks['pick_index'].astype(int)

    catalog = pd.read_csv(CATALOG_FILE, sep='\t', engine='python')
    stations = pd.read_csv(STATION_FILE, delim_whitespace=True)
    reloc = pd.read_csv(RELOC_FILE, delim_whitespace=True)

    # ---------- 2. 数据规范 ----------
    assignments['pick_index'] = pd.to_numeric(assignments['pick_index'], errors='coerce')
    assignments = assignments[assignments['pick_index'].notna()].copy()
    assignments['pick_index'] = assignments['pick_index'].astype(int)
    assignments['event_index'] = assignments['event_index'].astype(str)
    catalog['event_index'] = catalog['event_index'].astype(str)
    merge_picks['station_id'] = merge_picks['station_id'].astype(str)
    stations['station_id'] = stations['station_id'].astype(str)

    if 'phase_amplitude' not in merge_picks.columns or 'phase_type' not in merge_picks.columns:
        raise RuntimeError("merge_picks 必须包含列 'phase_amplitude' 和 'phase_type'。")

    # ---------- 3. 主计算 ----------
    station_level_rows = []
    event_summary_rows = []
    stations_idx = stations.set_index('station_id')[['latitude', 'longitude']].to_dict(orient='index')
    event_ids = assignments['event_index'].unique()

    for ev_id in event_ids:
        pick_indices = assignments.loc[assignments['event_index'] == ev_id, 'pick_index'].values
        picks_ev = merge_picks[merge_picks['pick_index'].isin(pick_indices)].copy()

        # 仅取 S 相
        picks_ev_S = picks_ev[picks_ev['phase_type'].str.upper() == 'S']
        if picks_ev_S.empty:
            continue

        # 事件位置
        ev_row = catalog[catalog['event_index'] == ev_id]
        if ev_row.empty:
            continue
        ev_lat = float(ev_row.iloc[0]['latitude'])
        ev_lon = float(ev_row.iloc[0]['longitude'])

        grouped = (
            picks_ev_S.groupby('station_id', as_index=False)['phase_amplitude']
            .mean()
            .rename(columns={'phase_amplitude': 'mean_amp'})
        )

        M_group_list = []
        for _, g in grouped.iterrows():
            station_id = str(g['station_id'])
            mean_amp = g['mean_amp']
            if station_id not in stations_idx:
                continue
            st_lat = stations_idx[station_id]['latitude']
            st_lon = stations_idx[station_id]['longitude']

            dist_km = haversine(ev_lat, ev_lon, st_lat, st_lon)
            M_group = calc_ML(mean_amp, dist_km)

            station_level_rows.append({
                'event_index': ev_id,
                'station_id': station_id,
                'mean_amp': mean_amp,
                'dist_km': dist_km,
                'ML_group': M_group
            })
            if not np.isnan(M_group):
                M_group_list.append(M_group)

        if len(M_group_list) > 0:
            ML_final = np.mean(M_group_list)
        else:
            ML_final = np.nan

        event_summary_rows.append({
            'event_index': ev_id,
            'final_ML': ML_final,
            'station_group_count': len(M_group_list)
        })

    # ---------- 4. 保存结果 ----------
    df_station_groups = pd.DataFrame(station_level_rows)
    df_event_summary = pd.DataFrame(event_summary_rows)
    df_station_groups.to_csv(OUT_GROUP_FILE, index=False, float_format='%.6f')
    df_event_summary.to_csv(OUT_SUMMARY_FILE, index=False, float_format='%.6f')

    print(f"S相逐台站ML结果保存: {OUT_GROUP_FILE}")
    print(f"S相事件ML汇总结果保存: {OUT_SUMMARY_FILE}")

    # ---------- 5. 写回 reloc ----------
    if 'magnitude' not in reloc.columns:
        reloc['magnitude'] = np.nan
    reloc['magnitude_original'] = reloc['magnitude']
    reloc['id'] = reloc['id'].astype(str)
    df_event_summary['event_index'] = df_event_summary['event_index'].astype(str)
    mapping = dict(zip(df_event_summary['event_index'], df_event_summary['final_ML']))

    def _update_mag(row):
        eid = row['id']
        if eid in mapping and not pd.isna(mapping[eid]):
            return mapping[eid]
        else:
            return row['magnitude']

    reloc['magnitude'] = reloc.apply(_update_mag, axis=1)
    reloc.to_csv(OUT_UPDATED_RELOC, sep=' ', index=False, float_format='%.6f', na_rep='nan')
    print(f"✅ 更新后的 ML 震级写回文件: {OUT_UPDATED_RELOC}")

if __name__ == '__main__':
    main()

S相逐台站ML结果保存: event_station_results_Sphase_ML.csv
S相事件ML汇总结果保存: event_magnitude_summary_Sphase_ML.csv
✅ 更新后的 ML 震级写回文件: hypoDD_v2_1307_withmag_modified_final_Sphase_ML.reloc


### 合并reloc

In [ ]:
import pandas as pd
import glob

# 假设你的 6 个 reloc 文件都放在同一个目录下，并且以 .reloc 结尾
files = [
    '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_20228_11_raw/hypoDD_phasenet_2208_11_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2212raw/hypoDD_phasenet202212_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/phasenet_2023raw/hypoDD_phasenet2023raw_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/eqt_20228_11_raw/hypoDD_eqt2208_11raw_withmag_updated.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/eqt_2212raw/hypoDD_eqtraw2212_withmap.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/eqt_2023raw/hypoDD_eqt2023raw_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/tran_2208_11/hypoDD_tran2208_11raw_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/tran_2212raw/hypoDD_tran2212raw_withmag.reloc',
         '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/tran_2023raw/hypoDD_tran2023raw_withmag.reloc'
         ]

# 指定列名
cols = ['lat', 'lon', 'depth', 'year', 'month', 'day', 'hour', 'minute', 'second', 'magnitude']

# 读取并合并所有文件
df_list = []
for f in files:
    df = pd.read_csv(f, sep=r"\s+", usecols=cols, engine='python')
    df_list.append(df)

# for f in files[6:]:
#     df = pd.read_csv(f, sep="\t", usecols=cols, engine='python')
#     df_list.append(df)

# 合并为一个总表
data = pd.concat(df_list, ignore_index=True)

# 创建时间列
data['time'] = pd.to_datetime(
    data[['year', 'month', 'day', 'hour', 'minute', 'second']]
)

# 按 time 升序排序
data = data.sort_values('time').reset_index(drop=True)
# data = data[data['magnitude'] != -1.0]
# 导出为新的 reloc 文件
data.to_csv("/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/hypoDD_final_allevents.reloc", sep="\t", index=False)

### 写一个代码修改station_id，为2212的两个文件

In [31]:
import pandas as pd

file_name = '/Users/chouyuhin/_Workhome/Proj_Seismicgap/QJvalidation/eqt_2212raw/eqt_picks_raw2212_withamp.csv'
df = pd.read_csv(file_name, sep=",", engine='python')
print(df['station_id'].head())
print(df['station_id'].str.split('.'))
df['station_id'] = df['station_id'].str.split('.').str[1] # 去除多余空格
# print(df['station_id'])
print(df.head())
df.to_csv(file_name, sep=",", index=False)

0    QJ.QJ01.
1    QJ.QJ01.
2    QJ.QJ01.
3    QJ.QJ01.
4    QJ.QJ01.
Name: station_id, dtype: object
0       [QJ, QJ01, ]
1       [QJ, QJ01, ]
2       [QJ, QJ01, ]
3       [QJ, QJ01, ]
4       [QJ, QJ01, ]
            ...     
6769    [QJ, QJ10, ]
6770    [QJ, QJ10, ]
6771    [QJ, QJ10, ]
6772    [QJ, QJ10, ]
6773    [QJ, QJ10, ]
Name: station_id, Length: 6774, dtype: object
   index station_id                     time               start_time  \
0      0       QJ01  2022-12-01 02:00:18.430  2022-12-01 02:00:18.330   
1      1       QJ01  2022-12-01 02:00:20.690  2022-12-01 02:00:20.650   
2      2       QJ01  2022-12-01 04:40:06.660  2022-12-01 04:40:06.500   
3      3       QJ01  2022-12-01 04:40:08.290  2022-12-01 04:40:08.140   
4      0       QJ01  2022-12-01 06:57:15.900  2022-12-01 06:57:15.770   

                  end_time  probability phase  phase_amplitude  
0  2022-12-01 02:00:18.710     0.830987     P           1934.0  
1  2022-12-01 02:00:20.950     0.734499     S       